# Module 1: Absmax & Zero-Point Quantization

This notebook walks through the two foundational quantization schemes step by step.

**What you'll learn:**
- How absmax (symmetric) quantization works
- How zero-point (asymmetric) quantization works
- How to measure quantization error
- How to apply these to a real LLM (TinyLlama)

**Read first:** `docs/theory.md` sections 2-6

**Time:** ~30 minutes

In [ ]:
import sys
sys.path.insert(0, '..')  # allows importing from src/

import torch
import matplotlib.pyplot as plt
import numpy as np

from src.quantization.absmax import absmax_quantize, absmax_dequantize, absmax_quantize_per_channel
from src.quantization.zeropoint import zeropoint_quantize, zeropoint_dequantize

print('Setup complete!')

## Part 1: Absmax Quantization — Intuition

Start with a simple 1D example to build intuition.

In [ ]:
# Create a simple example tensor
x = torch.tensor([-2.5, -1.8, -0.9, 0.0, 0.7, 1.3, 2.0, 2.5])
print(f'Original tensor: {x}')
print(f'Shape: {x.shape}, dtype: {x.dtype}')
print(f'Memory: {x.element_size() * x.nelement()} bytes (float32 = 4 bytes each)')

# Apply absmax quantization
q, scale = absmax_quantize(x, bits=8)
print(f'\nQuantized (INT8): {q}')
print(f'Scale: {scale:.6f}')
print(f'Memory: {q.element_size() * q.nelement()} bytes (int8 = 1 byte each)')

# Dequantize
x_recovered = absmax_dequantize(q, scale)
print(f'\nRecovered: {x_recovered}')
print(f'Error:     {x - x_recovered}')

In [ ]:
# Visualize the mapping from float to integer and back
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(range(len(x)), x.numpy(), color='steelblue')
axes[0].set_title('Original Float32 values')
axes[0].set_ylabel('Value')
axes[0].axhline(y=0, color='black', linewidth=0.5)

axes[1].bar(range(len(q)), q.numpy(), color='orange')
axes[1].set_title('Quantized INT8 values\n(scale={:.4f})'.format(scale))
axes[1].set_ylabel('Integer Value')
axes[1].axhline(y=0, color='black', linewidth=0.5)

error = (x - x_recovered).abs()
axes[2].bar(range(len(error)), error.numpy(), color='red', alpha=0.7)
axes[2].set_title('Quantization Error\n(|original - recovered|)')
axes[2].set_ylabel('Absolute Error')
axes[2].axhline(y=scale, color='black', linestyle='--', label=f'scale={scale:.4f}')
axes[2].legend()

plt.tight_layout()
plt.show()
print(f'Max possible error = scale/2 = {scale/2:.6f}')

## Part 2: INT8 vs INT4 — Comparing Quantization Precision

In [ ]:
# Create a test tensor with typical weight-like values
torch.manual_seed(42)
W = torch.randn(32) * 0.15  # typical weight magnitude for small LLMs

# Quantize at different bit widths
q8, s8 = absmax_quantize(W, bits=8)
q4, s4 = absmax_quantize(W, bits=4)

W_int8 = absmax_dequantize(q8, s8)
W_int4 = absmax_dequantize(q4, s4)

err_int8 = (W - W_int8).abs()
err_int4 = (W - W_int4).abs()

print(f'INT8 scale: {s8:.5f}  (max error ≤ {s8/2:.5f})')
print(f'INT4 scale: {s4:.5f}  (max error ≤ {s4/2:.5f})')
print(f'INT4 scale is {s4/s8:.1f}x larger than INT8 scale (= {s4/s8:.1f}x worse precision)')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))

x_idx = range(len(W))
ax1.plot(x_idx, W.numpy(), 'b-o', label='Original FP32', markersize=4)
ax1.plot(x_idx, W_int8.numpy(), 'g--s', label='INT8 recovered', alpha=0.8, markersize=4)
ax1.plot(x_idx, W_int4.numpy(), 'r--^', label='INT4 recovered', alpha=0.8, markersize=4)
ax1.legend(); ax1.set_title('Original vs Recovered Weights'); ax1.grid(alpha=0.3)

ax2.bar(x_idx, err_int8.numpy(), alpha=0.7, color='green', label=f'INT8 error (mean={err_int8.mean():.5f})')
ax2.bar(x_idx, err_int4.numpy(), alpha=0.4, color='red', label=f'INT4 error (mean={err_int4.mean():.5f})')
ax2.legend(); ax2.set_title('Quantization Error per Weight'); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 3: Zero-Point vs Absmax — When Does Asymmetric Help?

In [ ]:
# Create a skewed distribution (like activations after ReLU)
# All values are positive — absmax wastes the negative side!
torch.manual_seed(0)
x_skewed = torch.abs(torch.randn(500)) * 0.5  # all positive

# Absmax quantization
q_abs, s_abs = absmax_quantize(x_skewed)
x_abs_recovered = absmax_dequantize(q_abs, s_abs)
err_abs = (x_skewed - x_abs_recovered).abs().mean().item()

# Zero-point quantization
q_zp, s_zp, zp = zeropoint_quantize(x_skewed)
x_zp_recovered = zeropoint_dequantize(q_zp, s_zp, zp)
err_zp = (x_skewed - x_zp_recovered).abs().mean().item()

print(f'Data range: [{x_skewed.min():.3f}, {x_skewed.max():.3f}] (all positive!)')
print(f'\nAbsmax:')
print(f'  Scale = {s_abs:.5f}')
print(f'  Mean abs error = {err_abs:.6f}')
print(f'  Integer range used: [{q_abs.min()}, {q_abs.max()}] (negative side wasted!)')
print(f'\nZero-Point:')
print(f'  Scale = {s_zp:.5f}, zero_point = {zp}')
print(f'  Mean abs error = {err_zp:.6f}')
print(f'  Integer range used: [{q_zp.min()}, {q_zp.max()}] (full range utilized!)')
print(f'\nZero-point is {err_abs/err_zp:.2f}x better for this skewed distribution!')

## Part 4: Apply to a Real LLM — TinyLlama Weight Distribution

In [ ]:
# Load TinyLlama
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
print(f'Loading {model_name}...')
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
print('Model loaded!')

# Show model size
from src.eval.metrics import measure_model_size_mb
size_mb = measure_model_size_mb(model)
print(f'Model size: {size_mb:.1f} MB')
print(f'Total parameters: {sum(p.nelement() for p in model.parameters()):,}')

In [ ]:
# Inspect weight distribution of one attention layer
W = model.model.layers[0].self_attn.q_proj.weight.data.float()
print(f'Layer: model.layers[0].self_attn.q_proj')
print(f'Shape: {W.shape}  ({W.shape[0]} output neurons × {W.shape[1]} input features)')
print(f'Stats: min={W.min():.4f}, max={W.max():.4f}, mean={W.mean():.4f}, std={W.std():.4f}')
print(f'Memory: {W.nelement() * 4 / 1024:.1f} KB as float32')
print(f'Memory: {W.nelement() * 1 / 1024:.1f} KB as int8')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of original weights
axes[0].hist(W.flatten().numpy(), bins=100, color='steelblue', alpha=0.8)
axes[0].set_title('TinyLlama q_proj weight distribution\n(FP16, layer 0)')
axes[0].set_xlabel('Weight value')
axes[0].set_ylabel('Count')
axes[0].axvline(x=0, color='red', linewidth=1, linestyle='--')

# Quantize and show quantized distribution
q, scale = absmax_quantize(W)
axes[1].hist(q.flatten().numpy(), bins=100, color='orange', alpha=0.8)
axes[1].set_title(f'After absmax INT8 quantization\n(scale={scale:.6f})')
axes[1].set_xlabel('Integer value (-127 to 127)')
axes[1].set_ylabel('Count')
axes[1].axvline(x=0, color='red', linewidth=1, linestyle='--')

plt.tight_layout()
plt.show()
print('Observation: TinyLlama weights ARE nearly symmetric — absmax is appropriate here!')

In [ ]:
# Quantize ALL linear layers and measure the memory impact
import copy
import torch.nn as nn

model_quantized = copy.deepcopy(model)
n_layers_quantized = 0
total_original_bytes = 0
total_quantized_bytes = 0

for name, module in model_quantized.named_modules():
    if isinstance(module, nn.Linear):
        W = module.weight.data.float()
        q, scale = absmax_quantize(W, bits=8)
        W_recovered = absmax_dequantize(q, scale)
        module.weight.data = W_recovered.half()  # back to fp16 storage
        
        total_original_bytes += W.nelement() * 4   # fp32 bytes
        total_quantized_bytes += q.nelement() * 1  # int8 bytes
        n_layers_quantized += 1

print(f'Quantized {n_layers_quantized} linear layers')
print(f'Original weight bytes (fp32):   {total_original_bytes/1024/1024:.1f} MB')
print(f'Quantized weight bytes (int8):  {total_quantized_bytes/1024/1024:.1f} MB')
print(f'Compression ratio: {total_original_bytes/total_quantized_bytes:.1f}x')
print()
print('Note: Our model still uses fp16 storage (we dequantize before storing).')
print('A production INT8 implementation stores int8 and dequantizes on-the-fly.')

In [ ]:
# Compare generation quality before and after quantization
prompt = 'The capital of France is'
inputs = tokenizer(prompt, return_tensors='pt')

print('=== BEFORE quantization ===')
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

print('\n=== AFTER absmax INT8 quantization ===')
with torch.no_grad():
    out = model_quantized.generate(**inputs, max_new_tokens=30, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

print('\n(Output should be very similar — INT8 loses minimal quality)')

## Part 5: Per-Channel vs Per-Tensor — Visualizing the Difference

In [ ]:
# Compare per-tensor vs per-channel quantization error on a real weight matrix
W = model.model.layers[0].self_attn.q_proj.weight.data.float()

# Per-tensor quantization
q_pt, s_pt = absmax_quantize(W)
W_pt = absmax_dequantize(q_pt, s_pt)
err_per_tensor = (W - W_pt).abs()

# Per-channel (per-row) quantization  
q_pc, scales_pc = absmax_quantize_per_channel(W)
from src.quantization.dequantize import dequantize_tensor
W_pc = dequantize_tensor(q_pc, scales_pc, scheme='absmax')
err_per_channel = (W - W_pc).abs()

print(f'Per-tensor scale: single value = {s_pt:.6f}')
print(f'Per-channel scales: {W.shape[0]} values, range=[{scales_pc.min():.6f}, {scales_pc.max():.6f}]')
print()
print(f'Per-tensor   mean error: {err_per_tensor.mean():.6f}')
print(f'Per-channel  mean error: {err_per_channel.mean():.6f}')
print(f'Per-channel is {err_per_tensor.mean()/err_per_channel.mean():.2f}x better!')

# Show error heatmap for a 32x32 slice
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
W_slice = W[:32, :32]

im1 = ax1.imshow(err_per_tensor[:32, :32].numpy(), cmap='hot')
ax1.set_title('Per-Tensor Error Heatmap\n(one global scale)')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(err_per_channel[:32, :32].numpy(), cmap='hot', vmax=err_per_tensor[:32,:32].max())
ax2.set_title('Per-Channel Error Heatmap\n(one scale per row)')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()
print('Lower values (darker) = less error = better!')

## Summary

What you learned in this notebook:

1. **Absmax quantization** maps `[-max, max]` to `[-127, 127]` using `scale = max/127`
2. **Zero-point quantization** maps `[min, max]` to `[0, 255]` using `scale = (max-min)/255`
3. **INT8 vs INT4:** INT4 scale is ~18x larger → much coarser → needs GPTQ compensation
4. **Per-channel is better** because each row of a weight matrix has different ranges
5. **Real models:** TinyLlama weights are near-symmetric, making absmax work well

**Next:** `02_gptq_walkthrough.ipynb` — how GPTQ fixes INT4 quality using Hessians